# Q1a: Forecast total number of vehicles entering the parking per day, for next 7 days

**INSTRUCTIONS TO USER:**
1. Download `parkingLot.csv` from the provided Kaggle link in your course material.
2. Place `parkingLot.csv` in the `Week 4/Assignment` directory.
3. Make sure you are using the `.venv` environment from `Week 4`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
# 1. Load Data
df = pd.read_csv('parkingLot.csv', dtype={'camera_id': str})
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Data Cleaning
# Filter for 'camera_id' == '001' (entry)
df = df[df['camera_id'] == '001']

# Filter out 12am to 5am (mall closed)
df = df[df['timestamp'].dt.hour >= 5]
# Handle maintenance (nulls) and weekend spikes (already handled implicitly by time series patterns)

df = df.dropna()

# Fix OCR errors (e.g. replacing 'O' with '0') if needed
df['vehicle_no'] = df['vehicle_no'].str.replace('O', '0')

# 3. Aggregate by day to get daily vehicle counts
daily_counts = df.set_index('timestamp').resample('D').size()

# 4. Train/Test Split
train = daily_counts[:-7]
test = daily_counts[-7:]

In [3]:
# 5. Modeling - ARIMA or ETS (Here using a placeholder ARIMA model)
model = ARIMA(train, order=(5,1,0))
model_fit = model.fit()

# 6. Forecasting for next 7 days
forecast = model_fit.forecast(steps=7)

# 7. Evaluation (MASE and MAPE)
mape = mean_absolute_percentage_error(test, forecast)
print(f"MAPE: {mape}")

# MASE implementation
def mase(y_true, y_pred, y_train):
    n = len(y_train)
    d = np.abs(np.diff(y_train)).sum() / (n-1)
    errors = np.abs(y_true - y_pred)
    return errors.mean() / d
print(f"MASE: {mase(test, forecast, train)}")

MAPE: 0.08131910724481137
MASE: 0.9359360709523671
